In [109]:
pip install pandas wbgapi pyarrow

Note: you may need to restart the kernel to use updated packages.


In [110]:
import pandas as pd
import wbgapi as wb
from pathlib import Path

In [111]:
# PAÍSES
# =====================================================

countries = pd.DataFrame(list(wb.economy.list()))
countries = countries[countries["aggregate"] == False]

country_codes = countries["id"].tolist()

print(f"{len(country_codes)} países encontrados")

# INDICADORES
# =====================================================

INDICATORS = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "real_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "inflation_rate": "FP.CPI.TOTL.ZG",
    "compulsory_education_years": "SE.COM.DURS",
    "population_65_plus_pct": "SP.POP.65UP.TO.ZS",
    "unemployment_rate": "SL.UEM.TOTL.ZS",
    "government_expenditure_pct_gdp": "GC.XPN.TOTL.GD.ZS",
    "government_revenue_pct_gdp": "GC.REV.XGRT.GD.ZS",
    "gdp_deflator_inflation": "NY.GDP.DEFL.KD.ZG",
    "age_dependency_ratio": "SP.POP.DPND",
    "gross_capital_formation_pct_gdp": "NE.GDI.TOTL.ZS",
    "total_reserves_months_imports": "FI.RES.TOTL.MO",
    "savings_pct_gdp": "NY.GNS.ICTR.ZS",
    "exports_pct_gdp": "NE.EXP.GNFS.ZS"
}

# FUNÇÃO DE DOWNLOAD
# =====================================================

def download_indicator(indicator_code, variable_name):

    print(f"Baixando {variable_name}...")

    df = wb.data.DataFrame(
        indicator_code,
        economy=country_codes,
        time=range(2010, 2026),
        labels=True
    )

    df = df.reset_index()

    df = df.melt(
        id_vars=["economy", "Country"],
        var_name="year",
        value_name=variable_name
    )

    df["year"] = (
        df["year"]
        .str.replace("YR", "", regex=False)
        .astype(int)
    )

    return df


# PRIMEIRO INDICADOR
# =====================================================

first_var = list(INDICATORS.keys())[0]
first_code = INDICATORS[first_var]

panel = download_indicator(first_code, first_var)

# MERGE DOS DEMAIS
# =====================================================

for var, code in list(INDICATORS.items())[1:]:

    temp = download_indicator(code, var)

    panel = panel.merge(
        temp[
            ["economy", "year", var]
        ],
        on=["economy", "year"],
        how="left"
    )

# RENOMEIA
# =====================================================

panel = panel.rename(columns={
    "economy": "country_code",
    "Country": "country"
})

# ORDENA
# =====================================================

panel = panel.sort_values(
    ["country", "year"]
).reset_index(drop=True)

# RESULTADO
# =====================================================

print(panel.head())

print("\nShape:")
print(panel.shape)

print("\nMissing Values:")
print(
    panel.isna()
    .mean()
    .sort_values(ascending=False)
)

# SALVA
# =====================================================

panel.to_csv(
    "wdi_fiscal_risk_panel.csv",
    index=False
)

print("\nArquivo salvo!")

217 países encontrados
Baixando gdp_per_capita...
Baixando real_gdp_growth...
Baixando inflation_rate...
Baixando compulsory_education_years...
Baixando population_65_plus_pct...
Baixando unemployment_rate...
Baixando government_expenditure_pct_gdp...
Baixando government_revenue_pct_gdp...
Baixando gdp_deflator_inflation...
Baixando age_dependency_ratio...
Baixando gross_capital_formation_pct_gdp...
Baixando total_reserves_months_imports...
Baixando savings_pct_gdp...
Baixando exports_pct_gdp...
  country_code      country  year  gdp_per_capita  real_gdp_growth  \
0          AFG  Afghanistan  2010      560.621505        14.362441   
1          AFG  Afghanistan  2011      606.694676         0.426355   
2          AFG  Afghanistan  2012      651.417134        12.752287   
3          AFG  Afghanistan  2013      637.087099         5.600745   
4          AFG  Afghanistan  2014      625.054942         2.724543   

   inflation_rate  compulsory_education_years  population_65_plus_pct  \
0    

In [112]:
coverage = (
    panel
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                        0.00
country                             0.00
year                                0.00
population_65_plus_pct              6.25
age_dependency_ratio                6.25
gdp_per_capita                      9.45
real_gdp_growth                     9.99
gdp_deflator_inflation             10.20
compulsory_education_years         10.63
unemployment_rate                  14.23
inflation_rate                     20.42
exports_pct_gdp                    22.44
gross_capital_formation_pct_gdp    25.58
total_reserves_months_imports      28.97
savings_pct_gdp                    31.77
government_revenue_pct_gdp         43.98
government_expenditure_pct_gdp     46.98
dtype: float64


In [113]:
missing_by_country = (
    panel
    .isna()
    .groupby(panel["country"])
    .mean()
    .mul(100)
    .round(2)
)

print(missing_by_country.head())

                country_code  country  year  gdp_per_capita  real_gdp_growth  \
country                                                                        
Afghanistan              0.0      0.0   0.0           12.50            12.50   
Albania                  0.0      0.0   0.0            6.25             6.25   
Algeria                  0.0      0.0   0.0            6.25             6.25   
American Samoa           0.0      0.0   0.0           18.75            18.75   
Andorra                  0.0      0.0   0.0            6.25             6.25   

                inflation_rate  compulsory_education_years  \
country                                                      
Afghanistan               6.25                         0.0   
Albania                   6.25                         0.0   
Algeria                   6.25                         0.0   
American Samoa          100.00                       100.0   
Andorra                 100.00                         0.0   

    

In [114]:
# Colunas que não são variáveis
id_cols = ["country", "country_code", "year"]

vars_cols = [c for c in panel.columns if c not in id_cols]

# Percentual de missing por país e variável
missing_country_var = (
    panel
    .groupby("country_code")[vars_cols]
    .apply(lambda x: x.isna().mean())
)

# Países que possuem alguma variável 100% nula
countries_to_drop = (
    missing_country_var
    .eq(1)
    .any(axis=1)
)

countries_to_drop = countries_to_drop[
    countries_to_drop
].index.tolist()

print(f"Países removidos: {len(countries_to_drop)}")
print(countries_to_drop)

Países removidos: 99
['ABW', 'AND', 'ASM', 'ATG', 'BDI', 'BEN', 'BFA', 'BMU', 'BOL', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CHI', 'CHN', 'CIV', 'COM', 'CUB', 'CUW', 'CYM', 'DJI', 'DMA', 'DZA', 'ERI', 'FJI', 'FRO', 'FSM', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRD', 'GRL', 'GUM', 'GUY', 'HKG', 'HTI', 'IDN', 'IMN', 'IRN', 'JAM', 'JPN', 'KHM', 'KIR', 'KNA', 'KWT', 'LBR', 'LBY', 'LCA', 'LIE', 'MAF', 'MCO', 'MHL', 'MLI', 'MMR', 'MNE', 'MNP', 'MOZ', 'MRT', 'NCL', 'NER', 'NGA', 'NRU', 'OMN', 'PAK', 'PLW', 'PNG', 'PRI', 'PRK', 'PSE', 'PYF', 'QAT', 'SEN', 'SLB', 'SLE', 'SMR', 'SOM', 'SSD', 'STP', 'SUR', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'TKM', 'TTO', 'TUV', 'VCT', 'VEN', 'VGB', 'VIR', 'VNM', 'VUT', 'XKX', 'YEM']


In [115]:
panel_clean = panel[
    ~panel["country_code"].isin(countries_to_drop)
].copy()

print(panel.shape)
print(panel_clean.shape)

(3472, 17)
(1888, 17)


In [116]:
panel_clean = panel_clean.loc[
    panel_clean["country_code"].str.strip() != "TLS"
].copy()

In [117]:
panel_clean = panel[
    ~panel["country_code"].isin(countries_to_drop)
].copy()

print(panel.shape)
print(panel_clean.shape)

(3472, 17)
(1888, 17)


In [118]:
panel_clean = (
    panel_clean
    .loc[panel_clean["country_code"] != "TLS"]
    .reset_index(drop=True)
)

In [119]:
coverage = (
    panel_clean
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                        0.00
country                             0.00
year                                0.00
compulsory_education_years          0.27
unemployment_rate                   0.48
population_65_plus_pct              6.25
age_dependency_ratio                6.25
gdp_per_capita                      6.41
gdp_deflator_inflation              6.41
real_gdp_growth                     6.41
inflation_rate                      8.07
exports_pct_gdp                     8.65
gross_capital_formation_pct_gdp     8.76
total_reserves_months_imports       9.19
savings_pct_gdp                    10.68
government_revenue_pct_gdp         16.72
government_expenditure_pct_gdp     21.69
dtype: float64


In [120]:
id_cols = ["country", "country_code", "year"]

vars_cols = [
    c for c in panel_clean.columns
    if c not in id_cols
]

country_medians = (
    panel_clean
    .groupby("country_code")[vars_cols]
    .median()
    .reset_index()
)

print(country_medians.head())

  country_code  gdp_per_capita  real_gdp_growth  inflation_rate  \
0          AFG      523.775993         2.263629        4.673996   
1          AGO     2916.136633         2.102753       17.080954   
2          ALB     5006.360130         2.973154        2.028060   
3          ARE    47135.359194         4.552786        1.617488   
4          ARG    12704.283182        -1.026420       53.548304   

   compulsory_education_years  population_65_plus_pct  unemployment_rate  \
0                         9.0                2.356536            11.1855   
1                        10.0                2.725609            16.4845   
2                         9.0               13.305391            12.8400   
3                        12.0                1.501096             2.2795   
4                        14.0               11.307125             7.4225   

   government_expenditure_pct_gdp  government_revenue_pct_gdp  \
0                       43.126009                   10.646287   
1         

In [121]:
id_cols = ["country", "country_code", "year"]

vars_cols = [
    c for c in panel_clean.columns
    if c not in id_cols
]

panel_imputed = panel_clean.copy()

panel_imputed[vars_cols] = (
    panel_imputed
    .groupby("country_code")[vars_cols]
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [122]:
coverage = (
    panel_imputed
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                       0.0
country                            0.0
year                               0.0
gdp_per_capita                     0.0
real_gdp_growth                    0.0
inflation_rate                     0.0
compulsory_education_years         0.0
population_65_plus_pct             0.0
unemployment_rate                  0.0
government_expenditure_pct_gdp     0.0
government_revenue_pct_gdp         0.0
gdp_deflator_inflation             0.0
age_dependency_ratio               0.0
gross_capital_formation_pct_gdp    0.0
total_reserves_months_imports      0.0
savings_pct_gdp                    0.0
exports_pct_gdp                    0.0
dtype: float64


In [123]:
wdi_countries = set(panel_imputed["country_code"].unique())
print(f"WDI: {len(wdi_countries)} países")

WDI: 117 países


In [124]:
panel_imputed.to_csv(
    "wdi_panel.csv",
    index=False
)